## Missing Cross Channel Reference MiniLM recovery 

In [1]:
import importlib,json
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

import src.paths as paths
import src.defects as defects
import src.representations as representations

importlib.reload(paths)
importlib.reload(defects)
importlib.reload(representations)

from src.paths import get_project_paths
from src.defects import make_missing_cross_channel_plan,inject_missing_cross_channel_reference
from src.representations import serialize_campaign_dataset,encode_serialized_campaigns,split_campaign_into_chunks

PATHS=get_project_paths()
DATA_DIR=PATHS["processed"]
RESULTS_DIR=PATHS["results"]

SEVERITIES=[.10,.30,.60]

In [2]:
# load formal data and split
clean_campaigns=pd.read_csv(
    DATA_DIR/"clean_structured_campaigns.csv",
    parse_dates=["timestamp"]
)

split_df=pd.read_csv(
    DATA_DIR/"structural_campaign_split.csv"
)

eval_ids=set(
    split_df.loc[
        split_df["split"]=="evaluation",
        "campaign_id"
    ]
)

clean_eval=(
    clean_campaigns[
        clean_campaigns["campaign_id"].isin(eval_ids)
    ]
    .copy()
    .reset_index(drop=True)
)

print("Evaluation campaigns:",clean_eval["campaign_id"].nunique())
print("Messages:",len(clean_eval))

assert clean_eval["campaign_id"].nunique()==300

Evaluation campaigns: 300
Messages: 2090


In [3]:
# recover exact defect plan
missing_plan=make_missing_cross_channel_plan(
    clean_eval,
    seed=42
)

print("Plan rows:",len(missing_plan))
print("Unique campaigns:",missing_plan["campaign_id"].nunique())

display(missing_plan.head())

assert len(missing_plan)==300
assert missing_plan["campaign_id"].nunique()==300

Plan rows: 300
Unique campaigns: 300


,campaign_id,target_message_id,target_message_order,reference_id,reference_message_order,target_channel,original_reference_channel,replacement_channel,sender,original_text,severity_rank
0,C0003,C0003_M03,3,C0003_M01,1,email,sms,whatsapp,advisor_07,Following up here on the sms message I sent ea...,27
1,C0004,C0004_M03,3,C0004_M01,1,sms,email,whatsapp,advisor_17,Following up here on the email message I sent ...,256
2,C0006,C0006_M03,3,C0006_M01,1,whatsapp,sms,email,advisor_06,Following up here on the sms message I sent ea...,46
3,C0008,C0008_M03,3,C0008_M01,1,sms,email,whatsapp,advisor_15,Following up here on the email message I sent ...,41
4,C0010,C0010_M03,3,C0010_M01,1,sms,email,whatsapp,advisor_17,Following up here on the email message I sent ...,141


In [4]:
# reconstruct 3 defect conditions
missing_conditions={"clean":clean_eval.copy()}
missing_audits={}

for severity in SEVERITIES:
    defective,audit=inject_missing_cross_channel_reference(
        clean_eval,
        missing_plan,
        severity=severity
    )

    missing_conditions[severity]=defective
    missing_audits[severity]=audit

    print(
        f"{severity:.0%} | affected =",
        len(audit)
    )

10% | affected = 30
30% | affected = 90
60% | affected = 180


In [5]:
# validate nesting and exact ffected counts
affected={
    s:set(missing_audits[s]["campaign_id"])
    for s in SEVERITIES
}

assert affected[.10].issubset(affected[.30])
assert affected[.30].issubset(affected[.60])

assert len(affected[.10])==30
assert len(affected[.30])==90
assert len(affected[.60])==180

print("Nested affected sets passed.")

Nested affected sets passed.


In [6]:
# compara against frozen affected ids
affected_path=DATA_DIR/"missing_reference_affected_campaign_ids.csv"

saved_affected=pd.read_csv(affected_path)

for severity in SEVERITIES:
    reconstructed=set(
        missing_audits[severity]["campaign_id"].astype(str)
    )

    frozen=set(
        saved_affected.loc[
            np.isclose(saved_affected["severity"],severity),
            "campaign_id"
        ].astype(str)
    )

    print(
        f"{severity:.0%} | "
        f"reconstructed={len(reconstructed)} | "
        f"frozen={len(frozen)} | "
        f"exact match={reconstructed==frozen}"
    )

    assert reconstructed==frozen

10% | reconstructed=30 | frozen=30 | exact match=True
30% | reconstructed=90 | frozen=90 | exact match=True
60% | reconstructed=180 | frozen=180 | exact match=True


In [7]:
# validate that only text changed
NON_TEXT_COLUMNS=[
    "campaign_id",
    "message_id",
    "message_order",
    "timestamp",
    "channel",
    "sender",
    "recipient",
    "entity_id",
    "reference_id",
    "parent_message_id"
]

for severity in SEVERITIES:
    defective=missing_conditions[severity]

    assert (
        clean_eval[NON_TEXT_COLUMNS]
        .reset_index(drop=True)
        .equals(
            defective[NON_TEXT_COLUMNS]
            .reset_index(drop=True)
        )
    )

    clean_indexed=clean_eval.set_index("message_id")
    defective_indexed=defective.set_index("message_id")

    changed=(
        clean_indexed["text"]
        !=defective_indexed["text"]
    )

    changed_messages=changed[changed].index

    changed_campaigns=(
        clean_indexed.loc[
            changed_messages,
            "campaign_id"
        ]
        .value_counts()
    )

    assert (changed_campaigns==1).all()
    assert len(changed_messages)==int(severity*300)

    print(
        f"{severity:.0%} | "
        f"changed messages={len(changed_messages)}"
    )

print("Non-text preservation passed.")

10% | changed messages=30
30% | changed messages=90
60% | changed messages=180
Non-text preservation passed.


In [8]:
# recreate serialized campaigns
missing_serialized={
    key:serialize_campaign_dataset(df)
    for key,df in missing_conditions.items()
}

for key,df in missing_serialized.items():
    print(key,df.shape)

    assert df.shape==(300,2)

clean (300, 2)
0.1 (300, 2)
0.3 (300, 2)
0.6 (300, 2)


In [9]:
# load frozen minilm config
with open(
    RESULTS_DIR/"semantic_embedding_configuration.json"
) as f:
    semantic_config=json.load(f)

MINILM_MODEL_NAME=semantic_config["model"]
MINILM_REVISION=semantic_config["revision"]

print("Model:",MINILM_MODEL_NAME)
print("Revision:",MINILM_REVISION)

minilm_model=SentenceTransformer(
    MINILM_MODEL_NAME,
    revision=MINILM_REVISION
)

print(
    "Dimension:",
    minilm_model.get_sentence_embedding_dimension()
)
print(
    "Max sequence length:",
    minilm_model.max_seq_length
)

Model: sentence-transformers/all-MiniLM-L6-v2
Revision: 1110a243fdf4706b3f48f1d95db1a4f5529b4d41
Dimension: 384
Max sequence length: 256


In [10]:
# reconstruct minilm embeddings
reconstructed_minilm={}

for condition in ["clean",.10,.30,.60]:
    X,runtime=encode_serialized_campaigns(
        missing_serialized[condition],
        minilm_model,
        batch_size=64
    )

    reconstructed_minilm[condition]=X

    print(
        condition,
        X.shape,
        f"{runtime:.2f}s"
    )

Token indices sequence length is longer than the specified maximum sequence length for this model (259 > 256). Running this sequence through the model will result in indexing errors


Campaigns: 300
Chunks to encode: 308


Batches:   0%|          | 0/5 [00:00<?, ?it/s]

clean (300, 384) 32.95s
Campaigns: 300
Chunks to encode: 308


Batches:   0%|          | 0/5 [00:00<?, ?it/s]

0.1 (300, 384) 32.54s
Campaigns: 300
Chunks to encode: 308


Batches:   0%|          | 0/5 [00:00<?, ?it/s]

0.3 (300, 384) 33.32s
Campaigns: 300
Chunks to encode: 309


Batches:   0%|          | 0/5 [00:00<?, ?it/s]

0.6 (300, 384) 53.10s


In [11]:
# load official embeddings
official_minilm={
    "clean":np.load(
        DATA_DIR/"missing_reference_embeddings_clean.npy"
    ),
    .10:np.load(
        DATA_DIR/"missing_reference_embeddings_10pct.npy"
    ),
    .30:np.load(
        DATA_DIR/"missing_reference_embeddings_30pct.npy"
    ),
    .60:np.load(
        DATA_DIR/"missing_reference_embeddings_60pct.npy"
    )
}

for key,X in official_minilm.items():
    print(key,X.shape)

clean (300, 384)
0.1 (300, 384)
0.3 (300, 384)
0.6 (300, 384)


In [12]:
# exact recovery check
comparison=[]

for condition in ["clean",.10,.30,.60]:
    A=reconstructed_minilm[condition]
    B=official_minilm[condition]

    comparison.append({
        "condition":condition,
        "shape_match":A.shape==B.shape,
        "max_abs_diff":float(
            np.max(np.abs(A-B))
        ),
        "mean_abs_diff":float(
            np.mean(np.abs(A-B))
        ),
        "allclose":bool(
            np.allclose(
                A,
                B,
                atol=1e-6,
                rtol=1e-5
            )
        )
    })

comparison_df=pd.DataFrame(comparison)

display(comparison_df)

assert comparison_df["shape_match"].all()
assert comparison_df["allclose"].all()

print("Exact Missing source conditions successfully recovered.")

,condition,shape_match,max_abs_diff,mean_abs_diff,allclose
0,clean,True,0.0,0.0,True
1,0.1,True,0.0,0.0,True
2,0.3,True,0.0,0.0,True
3,0.6,True,0.0,0.0,True


Exact Missing source conditions successfully recovered.


In [13]:
# save recovered formal plan
PLAN_DIR=DATA_DIR/"formal_defect_plans"
PLAN_DIR.mkdir(parents=True,exist_ok=True)

missing_plan.to_csv(
    PLAN_DIR/"missing_cross_channel_formal_plan_recovered.csv",
    index=False
)

print("Recovered Missing plan saved.")

Recovered Missing plan saved.


BGE

In [14]:
# load the frozen bge config from entity
BGE_DIR=DATA_DIR/"robustness_bge"

with open(
    BGE_DIR/"entity_bge_configuration.json"
) as f:
    bge_config=json.load(f)

BGE_MODEL_NAME=bge_config["model"]
BGE_MODEL_REVISION=bge_config["revision"]
BGE_BATCH_SIZE=bge_config["batch_size"]

bge_model=SentenceTransformer(
    BGE_MODEL_NAME,
    revision=BGE_MODEL_REVISION
)

print("Model:",BGE_MODEL_NAME)
print("Revision:",BGE_MODEL_REVISION)
print(
    "Dimension:",
    bge_model.get_sentence_embedding_dimension()
)
print(
    "Max sequence length:",
    bge_model.max_seq_length
)

Model: BAAI/bge-base-en-v1.5
Revision: a5beb1e3e68b9ab74eb54cfd186867f64f240e1a
Dimension: 768
Max sequence length: 512


In [15]:
# verify that clean campaigns ids equal entity 
entity_bge_ids=pd.read_csv(
    BGE_DIR/"entity_bge_campaign_ids.csv"
)["campaign_id"].astype(str).to_numpy()

missing_clean_ids=(
    missing_serialized["clean"]["campaign_id"]
    .astype(str)
    .to_numpy()
)

print(
    "Same clean campaign IDs:",
    np.array_equal(
        entity_bge_ids,
        missing_clean_ids
    )
)

assert np.array_equal(
    entity_bge_ids,
    missing_clean_ids
)

Same clean campaign IDs: True


In [16]:
# validate bge chunking for missing
entity_bge_ids=pd.read_csv(
    BGE_DIR/"entity_bge_campaign_ids.csv"
)["campaign_id"].astype(str).to_numpy()

missing_clean_ids=(
    missing_serialized["clean"]["campaign_id"]
    .astype(str)
    .to_numpy()
)

print(
    "Same clean campaign IDs:",
    np.array_equal(
        entity_bge_ids,
        missing_clean_ids
    )
)

assert np.array_equal(
    entity_bge_ids,
    missing_clean_ids
)

Same clean campaign IDs: True


In [17]:
# generate missing bge embeddings 
missing_bge={}
missing_bge_runtime={}

for condition in ["clean",.10,.30,.60]:
    X,runtime=encode_serialized_campaigns(
        missing_serialized[condition],
        bge_model,
        batch_size=BGE_BATCH_SIZE
    )

    missing_bge[condition]=X
    missing_bge_runtime[condition]=runtime

    print(
        condition,
        X.shape,
        f"{runtime:.2f}s"
    )

Campaigns: 300
Chunks to encode: 300


Batches:   0%|          | 0/10 [00:00<?, ?it/s]

clean (300, 768) 187.11s
Campaigns: 300
Chunks to encode: 300


Batches:   0%|          | 0/10 [00:00<?, ?it/s]

0.1 (300, 768) 190.82s
Campaigns: 300
Chunks to encode: 300


Batches:   0%|          | 0/10 [00:00<?, ?it/s]

0.3 (300, 768) 179.67s
Campaigns: 300
Chunks to encode: 300


Batches:   0%|          | 0/10 [00:00<?, ?it/s]

0.6 (300, 768) 213.33s


In [18]:
# validate bge arrays 
for condition,X in missing_bge.items():
    assert X.shape[0]==300
    assert X.shape[1]==bge_model.get_sentence_embedding_dimension()
    assert np.isfinite(X).all()

    norms=np.linalg.norm(X,axis=1)

    assert np.allclose(
        norms,
        1.0,
        atol=1e-5
    )

print("All Missing BGE embeddings passed validation.")

All Missing BGE embeddings passed validation.


In [19]:
# confirm shared clean BGE with Entity
entity_bge_clean=np.load(
    BGE_DIR/"entity_bge_clean.npy"
)

print(
    "Max clean BGE difference:",
    np.max(
        np.abs(
            missing_bge["clean"]
            -entity_bge_clean
        )
    )
)

assert np.allclose(
    missing_bge["clean"],
    entity_bge_clean,
    atol=1e-6,
    rtol=1e-5
)

print("Entity and Missing share the same clean BGE representation.")

Max clean BGE difference: 0.0
Entity and Missing share the same clean BGE representation.


In [20]:
# verify changed ids under bge 
for severity in SEVERITIES:
    distances=np.linalg.norm(
        missing_bge[severity]
        -missing_bge["clean"],
        axis=1
    )

    changed_ids=set(
        missing_serialized["clean"].loc[
            distances>1e-8,
            "campaign_id"
        ]
    )

    expected_ids=set(
        missing_audits[severity]["campaign_id"]
    )

    print(
        f"{severity:.0%} | "
        f"changed={len(changed_ids)} | "
        f"expected={len(expected_ids)} | "
        f"exact match={changed_ids==expected_ids}"
    )

    assert changed_ids==expected_ids

10% | changed=30 | expected=30 | exact match=True
30% | changed=90 | expected=90 | exact match=True
60% | changed=180 | expected=180 | exact match=True


In [21]:
# save missing artifacts 
names={
    "clean":"missing_reference_bge_clean.npy",
    .10:"missing_reference_bge_10pct.npy",
    .30:"missing_reference_bge_30pct.npy",
    .60:"missing_reference_bge_60pct.npy"
}

for condition,name in names.items():
    np.save(
        BGE_DIR/name,
        missing_bge[condition]
    )

missing_serialized["clean"][["campaign_id"]].to_csv(
    BGE_DIR/"missing_reference_bge_campaign_ids.csv",
    index=False
)

print("Missing BGE artifacts saved.")

Missing BGE artifacts saved.


In [22]:
# save config 
missing_bge_config={
    "scenario":"Missing Cross-Channel Reference",
    "model":BGE_MODEL_NAME,
    "revision":BGE_MODEL_REVISION,
    "embedding_dimension":int(
        bge_model.get_sentence_embedding_dimension()
    ),
    "normalized":True,
    "batch_size":BGE_BATCH_SIZE,
    "campaign_serialization":"same as formal MiniLM experiment",
    "chunking":"same message-boundary procedure as formal MiniLM experiment",
    "aggregation":"token-weighted chunk mean followed by L2 normalization",
    "source_conditions_recovered_exactly":True,
    "shared_clean_with_entity":True,
    "minilm_reconstruction_max_abs_diff":float(
        comparison_df["max_abs_diff"].max()
    )
}

with open(
    BGE_DIR/"missing_reference_bge_configuration.json",
    "w"
) as f:
    json.dump(
        missing_bge_config,
        f,
        indent=4
    )

print("Missing BGE configuration saved.")

Missing BGE configuration saved.
